In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import re
import random
import warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from transformers import AutoTokenizer, AutoModelForMultipleChoice, get_linear_schedule_with_warmup
import joblib

warnings.filterwarnings('ignore')

def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

seed_everything(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using Device: {device}")

OPTS = ['A', 'B', 'C', 'D', 'E']
TRAIN_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv'
TEST_PATH = '/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv'

# Directory to save trained model artifacts
os.makedirs("models", exist_ok=True)

In [ ]:
def map_at_3(probs, y_true):
    top3 = np.argsort(-probs, axis=1)[:, :3]
    score = 0.0
    for i in range(len(y_true)):
        ranks = list(top3[i])
        if y_true[i] in ranks:
            score += 1.0 / (ranks.index(y_true[i]) + 1)
    return score / len(y_true)

def clean_text(text):
    if pd.isna(text):
        return ""
    text = str(text).lower()
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

print("Loading Data...")
train_df = pd.read_csv(TRAIN_PATH)
test_df = pd.read_csv(TEST_PATH)

encoder = LabelEncoder()
train_df['label'] = encoder.fit_transform(train_df['answer'])
joblib.dump(encoder, 'models/label_encoder.joblib')

for col in ['prompt', 'A', 'B', 'C', 'D', 'E']:
    train_df[f'{col}_clean'] = train_df[col].apply(clean_text)
    test_df[f'{col}_clean'] = test_df[col].apply(clean_text)

print(f"Train dataset shape: {train_df.shape}")

## MODEL 1: FROM-SCRATCH PYTORCH OPTIONSCORER

In [ ]:
print("\n--- Training Model 1: From-Scratch PyTorch OptionScorer ---")

word_vec = TfidfVectorizer(stop_words='english', max_features=4000, sublinear_tf=True, ngram_range=(1, 2), min_df=2)
char_vec = TfidfVectorizer(analyzer='char_wb', ngram_range=(3, 5), max_features=3000, sublinear_tf=True, min_df=2)
svd = TruncatedSVD(n_components=150, random_state=42)

full_corpus = (train_df['prompt'].fillna("").tolist() + test_df['prompt'].fillna("").tolist())
for c in OPTS:
    full_corpus += train_df[c].fillna("").tolist() + test_df[c].fillna("").tolist()

word_vec.fit(full_corpus)
char_vec.fit(full_corpus)
svd.fit(word_vec.transform(full_corpus))

joblib.dump(word_vec, 'models/scratch_word_vec.joblib')
joblib.dump(char_vec, 'models/scratch_char_vec.joblib')
joblib.dump(svd, 'models/scratch_svd.joblib')

def jaccard(a, b):
    sa, sb = set(a.lower().split()), set(b.lower().split())
    return len(sa & sb) / len(sa | sb) if sa and sb else 0.0

def build_scratch_features(df):
    n = len(df)
    feats = np.zeros((n, 5, 8), dtype=np.float32)
    prompts = df['prompt'].fillna("").tolist()
    opt_texts = {c: df[c].fillna("").tolist() for c in OPTS}

    p_word = word_vec.transform(prompts)
    p_char = char_vec.transform(prompts)
    p_svd = svd.transform(p_word)

    opt_word = {c: word_vec.transform(opt_texts[c]) for c in OPTS}
    opt_char = {c: char_vec.transform(opt_texts[c]) for c in OPTS}
    opt_svd = {c: svd.transform(opt_word[c]) for c in OPTS}

    for i in range(n):
        p_len = max(len(prompts[i]), 1)
        p_words = max(len(prompts[i].split()), 1)
        word_cos = np.array([cosine_similarity(p_word[i], opt_word[c][i])[0, 0] for c in OPTS])
        char_cos = [cosine_similarity(p_char[i], opt_char[c][i])[0, 0] for c in OPTS]
        svd_cos = [cosine_similarity(p_svd[i:i+1], opt_svd[c][i:i+1])[0, 0] for c in OPTS]
        jac = [jaccard(prompts[i], opt_texts[c][i]) for c in OPTS]
        len_ratio = [len(opt_texts[c][i]) / p_len for c in OPTS]
        wc_ratio = [len(opt_texts[c][i].split()) / p_words for c in OPTS]

        opt_vecs = [opt_word[c][i] for c in OPTS]
        centrality = [np.mean([cosine_similarity(opt_vecs[a], opt_vecs[b])[0, 0] for b in range(5) if b != a]) for a in range(5)]
        word_cos_z = (word_cos - word_cos.mean()) / (word_cos.std() + 1e-6)

        for k in range(5):
            feats[i, k] = [word_cos[k], char_cos[k], svd_cos[k], jac[k], len_ratio[k], wc_ratio[k], centrality[k], word_cos_z[k]]
    return feats

X_train_raw = build_scratch_features(train_df)
y_train_raw = train_df['label'].values

class OptionScorer(nn.Module):
    def __init__(self, in_features, hidden_dim=256, dropout=0.0):
        super(OptionScorer, self).__init__()
        self.net = nn.Sequential(
            nn.Linear(in_features, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim), nn.BatchNorm1d(hidden_dim), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim // 2), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(hidden_dim // 2, 1)
        )

    def forward(self, x):
        B, K, F = x.shape
        return self.net(x.reshape(B * K, F)).reshape(B, K)

num_feats = X_train_raw.shape[2]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

m2_oof_probs = np.zeros((len(train_df), 5))

for fold, (tr_idx, va_idx) in enumerate(skf.split(X_train_raw, y_train_raw)):
    Xtr, Xva = X_train_raw[tr_idx], X_train_raw[va_idx]
    ytr, yva = y_train_raw[tr_idx], y_train_raw[va_idx]

    mean = Xtr.reshape(-1, num_feats).mean(0)
    std = Xtr.reshape(-1, num_feats).std(0) + 1e-6
    joblib.dump((mean, std), f'models/scratch_scaler_fold{fold}.joblib')

    Xtr_n, Xva_n = (Xtr - mean) / std, (Xva - mean) / std

    Xtr_t = torch.tensor(Xtr_n, dtype=torch.float32).to(device)
    ytr_t = torch.tensor(ytr, dtype=torch.long).to(device)
    Xva_t = torch.tensor(Xva_n, dtype=torch.float32).to(device)

    model = OptionScorer(num_feats).to(device)
    optimizer = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    criterion = nn.CrossEntropyLoss()
    loader = DataLoader(torch.utils.data.TensorDataset(Xtr_t, ytr_t), batch_size=64, shuffle=True)

    for epoch in range(1, 251):
        model.train()
        for bx, by in loader:
            optimizer.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            optimizer.step()

    model.eval()
    with torch.no_grad():
        va_probs = torch.softmax(model(Xva_t), dim=1).cpu().numpy()

    torch.save(model.state_dict(), f'models/scratch_model_fold{fold}.pt')
    m2_oof_probs[va_idx] = va_probs
    print(f"Scratch OptionScorer | Fold {fold} MAP@3: {map_at_3(va_probs, yva):.4f}")

print(f"Model 1 Overall OOF MAP@3: {map_at_3(m2_oof_probs, y_train_raw):.4f}")


## TRANSFORMERS TRAINING HELPER (DeBERTa-v3-base & RoBERTa-base)

In [ ]:
class MCQDataset(Dataset):
    def __init__(self, df, tokenizer, max_length=256, has_labels=True):
        self.prompts = df['prompt'].fillna("").tolist()
        self.options = df[OPTS].fillna("").values.tolist()
        self.labels = df['label'].tolist() if has_labels else None
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.prompts)

    def __getitem__(self, idx):
        prompt = self.prompts[idx]
        opts = self.options[idx]
        encoded = self.tokenizer(
            [prompt] * 5, opts,
            truncation=True, max_length=self.max_length,
            padding='max_length', return_tensors='pt'
        )
        item = {k: v for k, v in encoded.items()}
        if self.labels is not None:
            item['labels'] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

def train_transformer_model(model_name, save_prefix):
    print(f"\n--- Training Transformer Model: {model_name} ---")
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.save_pretrained(f'models/{save_prefix}_tokenizer')

    oof_probs = np.zeros((len(train_df), 5))
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    for fold, (tr_idx, va_idx) in enumerate(skf.split(train_df, train_df['label'])):
        train_fold = train_df.iloc[tr_idx].reset_index(drop=True)
        val_fold = train_df.iloc[va_idx].reset_index(drop=True)

        train_dataset = MCQDataset(train_fold, tokenizer, max_length=256)
        val_dataset = MCQDataset(val_fold, tokenizer, max_length=256)
        train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)

        model = AutoModelForMultipleChoice.from_pretrained(model_name, torch_dtype=torch.float32).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)
        
        total_steps = (len(train_loader) // 2) * 3
        scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(total_steps * 0.1), num_training_steps=total_steps)
        scaler = torch.cuda.amp.GradScaler(enabled=(device.type == 'cuda'))

        for epoch in range(1, 4):
            model.train()
            optimizer.zero_grad()
            for step, batch in enumerate(train_loader):
                batch = {k: v.to(device) for k, v in batch.items()}
                with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                    outputs = model(**batch)
                    loss = outputs.loss / 2
                scaler.scale(loss).backward()

                if (step + 1) % 2 == 0:
                    scaler.unscale_(optimizer)
                    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                    scaler.step(optimizer)
                    scaler.update()
                    scheduler.step()
                    optimizer.zero_grad()

        # Validation Evaluation
        model.eval()
        val_probs = []
        with torch.no_grad():
            for batch in val_loader:
                batch = {k: v.to(device) for k, v in batch.items() if k != 'labels'}
                with torch.cuda.amp.autocast(enabled=(device.type == 'cuda')):
                    logits = model(**batch).logits
                val_probs.append(torch.softmax(logits, dim=1).float().cpu().numpy())
        val_probs = np.concatenate(val_probs, axis=0)
        oof_probs[va_idx] = val_probs

        model.save_pretrained(f'models/{save_prefix}_fold{fold}')
        print(f"{model_name} | Fold {fold} MAP@3: {map_at_3(val_probs, val_fold['label'].values):.4f}")

        del model
        torch.cuda.empty_cache()

    overall_map3 = map_at_3(oof_probs, train_df['label'].values)
    print(f"Overall OOF MAP@3 for {model_name}: {overall_map3:.4f}")
    return oof_probs

In [ ]:
# Train Model 2: DeBERTa-v3-base
m2_oof_probs = train_transformer_model("microsoft/deberta-v3-base", "deberta_v3")

# Train Model 3: RoBERTa-base
m3_oof_probs = train_transformer_model("roberta-base", "roberta")